In [2]:
# Notebook 02 — Preprocessing (PubMed 20k RCT)
# Cell 1 — Imports + paths + loader
import sys, json, re
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data.data_loader import parse_pubmed_rct_file  # adjust if your path differs

RAW_DIR  = PROJECT_ROOT / "data" / "raw" / "pubmed_20k_rct"
PROC_DIR = PROJECT_ROOT / "data" / "processed" / "pubmed_20k_rct"
PROC_DIR.mkdir(parents=True, exist_ok=True)

train_path = RAW_DIR / "train.txt"
dev_path   = RAW_DIR / "dev.txt"
test_path  = RAW_DIR / "test.txt"

train_raw = parse_pubmed_rct_file(train_path)
dev_raw   = parse_pubmed_rct_file(dev_path)
test_raw  = parse_pubmed_rct_file(test_path)

print("Raw shapes:", train_raw.shape, dev_raw.shape, test_raw.shape)
train_raw.head()


Raw shapes: (180040, 2) (30212, 2) (30135, 2)


,label,text
0,OBJECTIVE,To investigate the efficacy of 6 weeks of dail...
1,METHODS,A total of 125 patients with primary knee OA w...
2,METHODS,Outcome measures included pain reduction and i...
3,METHODS,Pain was assessed using the visual analog pain...
4,METHODS,Secondary outcome measures included the Wester...


In [3]:
# Cell 2 — Minimal cleaning function 
def clean_text(s: str) -> str:
    if s is None:
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)  # collapse whitespace
    return s

def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["label"] = out["label"].astype(str).str.strip()
    out["text"]  = out["text"].astype(str).map(clean_text)

    # drop empties (should be rare)
    out = out[out["text"].str.len() > 0].reset_index(drop=True)

    return out


In [4]:
# Cell 3 — Apply preprocessing to each split
train_df = preprocess_df(train_raw)
dev_df   = preprocess_df(dev_raw)
test_df  = preprocess_df(test_raw)

print("Processed shapes:", train_df.shape, dev_df.shape, test_df.shape)
train_df.head()


Processed shapes: (180040, 2) (30212, 2) (30135, 2)


,label,text
0,OBJECTIVE,To investigate the efficacy of 6 weeks of dail...
1,METHODS,A total of 125 patients with primary knee OA w...
2,METHODS,Outcome measures included pain reduction and i...
3,METHODS,Pain was assessed using the visual analog pain...
4,METHODS,Secondary outcome measures included the Wester...


In [5]:
# Cell 4 — Build a stable label mapping + add label_id
labels_sorted = sorted(train_df["label"].unique().tolist())
label2id = {label: i for i, label in enumerate(labels_sorted)}
id2label = {i: label for label, i in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)
dev_df["label_id"]   = dev_df["label"].map(label2id)
test_df["label_id"]  = test_df["label"].map(label2id)

# sanity check
assert train_df["label_id"].isna().sum() == 0
assert dev_df["label_id"].isna().sum() == 0
assert test_df["label_id"].isna().sum() == 0

label2id


{'BACKGROUND': 0, 'CONCLUSIONS': 1, 'METHODS': 2, 'OBJECTIVE': 3, 'RESULTS': 4}

In [6]:
# Cell 5 — Save processed splits + mapping artifacts
train_csv = PROC_DIR / "train.csv"
dev_csv   = PROC_DIR / "dev.csv"
test_csv  = PROC_DIR / "test.csv"

train_df.to_csv(train_csv, index=False)
dev_df.to_csv(dev_csv, index=False)
test_df.to_csv(test_csv, index=False)

with (PROC_DIR / "label2id.json").open("w", encoding="utf-8") as f:
    json.dump(label2id, f, indent=2)

with (PROC_DIR / "id2label.json").open("w", encoding="utf-8") as f:
    json.dump(id2label, f, indent=2)

print("Saved:")
print("-", train_csv)
print("-", dev_csv)
print("-", test_csv)
print("-", PROC_DIR / "label2id.json")
print("-", PROC_DIR / "id2label.json")


Saved:
- c:\Users\mansour\Documents\Clinical text classification\data\processed\pubmed_20k_rct\train.csv
- c:\Users\mansour\Documents\Clinical text classification\data\processed\pubmed_20k_rct\dev.csv
- c:\Users\mansour\Documents\Clinical text classification\data\processed\pubmed_20k_rct\test.csv
- c:\Users\mansour\Documents\Clinical text classification\data\processed\pubmed_20k_rct\label2id.json
- c:\Users\mansour\Documents\Clinical text classification\data\processed\pubmed_20k_rct\id2label.json


In [7]:
# Cell 6 — Preprocessing QC (quick checks)
def quick_qc(df: pd.DataFrame, name: str):
    print(f"\n== {name} ==")
    print("shape:", df.shape)
    print("nulls:", df.isna().sum().to_dict())
    print("label counts:")
    display(df["label"].value_counts())
    print("label_id range:", int(df["label_id"].min()), "->", int(df["label_id"].max()))
    display(df.head(3))

quick_qc(train_df, "TRAIN")
quick_qc(dev_df, "DEV")
quick_qc(test_df, "TEST")



== TRAIN ==
shape: (180040, 3)
nulls: {'label': 0, 'text': 0, 'label_id': 0}
label counts:


label
METHODS        59353
RESULTS        57953
CONCLUSIONS    27168
BACKGROUND     21727
OBJECTIVE      13839
Name: count, dtype: int64

label_id range: 0 -> 4


,label,text,label_id
0,OBJECTIVE,To investigate the efficacy of 6 weeks of dail...,3
1,METHODS,A total of 125 patients with primary knee OA w...,2
2,METHODS,Outcome measures included pain reduction and i...,2



== DEV ==
shape: (30212, 3)
nulls: {'label': 0, 'text': 0, 'label_id': 0}
label counts:


...

,label,text,label_id
0,BACKGROUND,IgE sensitization to Aspergillus fumigatus and...,0
1,BACKGROUND,It is not clear whether these patients would b...,0
2,OBJECTIVE,We sought to determine whether a 3-month cours...,3



== TEST ==
shape: (30135, 3)
nulls: {'label': 0, 'text': 0, 'label_id': 0}
label counts:


label
METHODS        9897
RESULTS        9713
CONCLUSIONS    4571
BACKGROUND     3621
OBJECTIVE      2333
Name: count, dtype: int64

label_id range: 0 -> 4


,label,text,label_id
0,BACKGROUND,This study analyzed liver function abnormaliti...,0
1,RESULTS,A post hoc analysis was conducted with the use...,4
2,RESULTS,Liver function tests ( LFTs ) were measured at...,4


In [8]:
# Cell 7 — Load back from disk to prove reproducibility
train_disk = pd.read_csv(PROC_DIR / "train.csv")
dev_disk   = pd.read_csv(PROC_DIR / "dev.csv")
test_disk  = pd.read_csv(PROC_DIR / "test.csv")

print("Disk shapes:", train_disk.shape, dev_disk.shape, test_disk.shape)
train_disk.head()


Disk shapes: (180040, 3) (30212, 3) (30135, 3)


,label,text,label_id
0,OBJECTIVE,To investigate the efficacy of 6 weeks of dail...,3
1,METHODS,A total of 125 patients with primary knee OA w...,2
2,METHODS,Outcome measures included pain reduction and i...,2
3,METHODS,Pain was assessed using the visual analog pain...,2
4,METHODS,Secondary outcome measures included the Wester...,2
